# API Gateway HTTP API with Boto3

Region: ap-south-1

In [ ]:
import boto3
from botocore.exceptions import ClientError

REGION = "ap-south-1"
DRY_RUN = True

apigw = boto3.client("apigatewayv2", region_name=REGION)
lambda_client = boto3.client("lambda", region_name=REGION)
sts = boto3.client("sts")

In [ ]:
API_NAME = "exam-http-api"
STAGE_NAME = "prod"
LAMBDA_ARN = "arn:aws:lambda:ap-south-1:123456789012:function:example"

In [ ]:
if DRY_RUN:
    api_id = "api-PLACEHOLDER"
    integration_id = "int-PLACEHOLDER"
    route_id = "route-PLACEHOLDER"
    print("[DRY RUN] create_api, create_integration, create_route, create_stage")
else:
    api = apigw.create_api(Name=API_NAME, ProtocolType="HTTP")
    api_id = api["ApiId"]
    integration = apigw.create_integration(
        ApiId=api_id,
        IntegrationType="AWS_PROXY",
        IntegrationUri=LAMBDA_ARN,
        PayloadFormatVersion="2.0"
    )
    integration_id = integration["IntegrationId"]
    route = apigw.create_route(
        ApiId=api_id,
        RouteKey="GET /",
        Target=f"integrations/{integration_id}",
    )
    route_id = route["RouteId"]
    apigw.create_stage(ApiId=api_id, StageName=STAGE_NAME, AutoDeploy=True)

In [ ]:
if DRY_RUN:
    print("[DRY RUN] add_lambda_permission and print endpoint")
else:
    account_id = sts.get_caller_identity()["Account"]
    source_arn = f"arn:aws:execute-api:{REGION}:{account_id}:{api_id}/*/*"
    lambda_client.add_permission(
        FunctionName=LAMBDA_ARN,
        StatementId="apigw-invoke",
        Action="lambda:InvokeFunction",
        Principal="apigateway.amazonaws.com",
        SourceArn=source_arn,
    )
    print(f"API endpoint: https://{api_id}.execute-api.{REGION}.amazonaws.com/{STAGE_NAME}/")

In [ ]:
if DRY_RUN:
    print("[DRY RUN] delete_route, delete_integration, delete_api")
else:
    apigw.delete_route(ApiId=api_id, RouteId=route_id)
    apigw.delete_integration(ApiId=api_id, IntegrationId=integration_id)
    apigw.delete_api(ApiId=api_id)